In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)

In [4]:
# module declaration
class CausalMean(nn.Module):
    def __init__(self, sequence_len):
        super().__init__()
        tril = torch.tril(
            input=torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool)
        )
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, x):
        T = x.size(1)
        scores = torch.zeros(size=(T, T))
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)
        return weights @ x

In [5]:
# model declaration
class BagOfWords(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size),
            CausalMean(sequence_len=sequence_len),
            nn.Linear(in_features=embedding_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=vocab_size),
        )

    def forward(self, x):
        return self.layers(x)

In [6]:
# model initialization
model = BagOfWords(
    vocab_size=vocab_size,
    embedding_size=embedding_size,
    sequence_len=sequence_len,
    hidden_size=hidden_size,
)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 27233


In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits = model(x_batch)
    train_loss = nn.functional.cross_entropy(
        input=train_logits.view(-1, vocab_size), target=y_batch.view(-1)
    )

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(
                input=val_logits.view(-1, vocab_size), target=y_val.view(-1)
            )
            print(
                f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}"
            )

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.1721673011779785, val loss = 4.173493385314941


Step 1000: train loss = 3.013653039932251, val loss = 3.059225082397461


Step 2000: train loss = 2.9842734336853027, val loss = 3.029104232788086


Step 3000: train loss = 2.9723689556121826, val loss = 3.0131895542144775


Step 4000: train loss = 2.979609966278076, val loss = 3.0150535106658936


Step 4999: train loss = 2.9742908477783203, val loss = 3.006094217300415


In [8]:
# prediction
with torch.inference_mode():
    output = ["R", "O", "M", "E", "O", ":", "\n"]
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits = model(i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor(
            [[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long
        )

In [9]:
print("".join(output))

ROMEO:
Wh weerelwa m leee eme asm toe erlwhvre'arosena,r v iee,l  oplawg onr ndf 
erqhwdrpo ieey n ofpTgtialrw bcsmuiiot u nsncIrahse are,tyto,e  lk t'.eh,o
a  ta
eat 
NhDEMIrI:CSvnwGwBe ilms dolta; hheyhaI emWes,  rp,el rt. w teaa!

eutSt eey,ey e euTlal lk   tHlfhm elhmnionrtshstte s enonrimg aooow?
 tdtoiehlBu!e we
uoi i ske se ;
nvA G Uaot AInL:hnvwrs,h,wue ufhr  n cIiiln oeo la cpattmlaaim;oe
si
ntte f eid
Arm unl n t,htgen CehserhonaR TSlsTaanho,tykw Cohad  nob eell i adeaeuwlit .t mu,enr ens
slsyao ne sao 
,hToht owT.  g
Ltmlliyaisepnoa EriWHM:'dits  
cahmsyussko m 'y    tahheOhitni sryueihdydse uilehfs
vttaeme,nu d e e aet;rgr trtsatsp  sh
oTtnttade  t rodbh beheoyehnoeaider yivms?eth ten
e   sno t
Ue Bsrl Hhns wwlrlv i e DaoSRbikIdIVCIZ:hikooe,cahrie;hE eutns tsd
 ma uI y 
sukeirn.
:t !rr 
ca
ntadf HuE NkohO xUfAunnevootIydNN  thrtae ir yo ikoO nl:nswyha uy,emuc;n p
weo f; bri en whmrrhe, tfwe
enoe  we-ey eiTyermtyae,ete sGn h  anv tttgooooo oru,'w e e  snro's ehourn eotohe ot